In [1]:
import os
import glob
import json
import numpy as np
import random
import wandb
import threading
import time
import cv2
import math
from scipy.spatial.transform import Rotation as R
from tqdm import tqdm 

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR # 상단에 추가
import kornia.augmentation as K

from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from transformers import AutoImageProcessor, AutoModel
from transformers.image_utils import load_image

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2"
is_available = torch.cuda.is_available()

if is_available:
    # 2. 사용 가능한 GPU 개수 확인
    gpu_count = torch.cuda.device_count()
    print(f"✅ 사용 가능한 GPU가 감지되었습니다.")
    print(f"GPU 개수: {gpu_count}개")

# 검색을 시작할 최상위 폴더 경로
search_dir = "../dataset/Converted_dataset"  # <-- 이곳에 실제 폴더 경로를 입력하세요.
json_files = glob.glob(os.path.join(search_dir, '**', '*.json'), recursive=True)

print(f"총 {len(json_files)}개의 JSON 파일을 찾았습니다.")

def load_sample(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        sample = json.load(f)
    obj = sample["objects"][0]
    meta = sample["meta"]
    sim_state = sample["sim_state"]
    return obj, meta, sim_state

def project_camframe_points(pts_cam, K, dist):
    """카메라 프레임 3D -> 픽셀 (extrinsic = I)"""
    rvec = np.zeros((3,1), np.float64)
    tvec = np.zeros((3,1), np.float64)
    pts = np.asarray(pts_cam, dtype=np.float64).reshape(-1,1,3)
    K = np.asarray(K, dtype=np.float64)
    dist = np.asarray(dist, dtype=np.float64)
    img_pts, _ = cv2.projectPoints(pts, rvec, tvec, K, dist)
    return img_pts.reshape(-1,2)

# 링크와 조인트 개수를 출력하는 기능을 추가합니다.
def visualize_json(json_path, show_ids=True, compute_reproj_error=True, point_radius=6, thickness=2):
    # 수정된 load_sample 함수를 호출합니다.
    obj, meta, sim_state = load_sample(json_path)
    
    image_path = meta["image_path"]
    
    # 링크와 조인트 개수를 계산합니다.
    num_keypoints = len(obj.get("keypoints", []))
    num_joints = len(sim_state.get("joints", []))
    
    # 계산된 정보를 출력합니다.
    print("-" * 50)
    print(f"Image Path: {image_path}")
    print(f"  - 🔗 링크 (Keypoints) 개수: {num_keypoints}개")
    print(f"  - 💪 조인트 (Joints) 개수: {num_joints}개")
    
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")

    # 이미지 로드
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 키포인트(픽셀) 수집
    names = []
    uv_from_json = []
    cam3d = []
    for kp in obj["keypoints"]:
        names.append(kp["name"])
        uv_from_json.append(kp["projected_location"])
        cam3d.append(kp["location"])
    uv_from_json = np.asarray(uv_from_json, dtype=np.float64)
    cam3d = np.asarray(cam3d, dtype=np.float64)

    # 재투영 (옵션)
    reproj_errs = None
    if compute_reproj_error and cam3d.size > 0:
        K = meta["K"]
        dist = meta.get("dist_coeffs", np.zeros(5))
        uv_reproj = project_camframe_points(cam3d, K, dist)
        reproj_errs = np.linalg.norm(uv_reproj - uv_from_json, axis=1)
        mean_err = float(reproj_errs.mean())
        print(f"[Reprojection] mean L2 error: {mean_err:.3f}px")

    # 드로잉
    overlay = img_rgb.copy()
    color = (255, 0, 255)  # magenta
    for i, (u, v) in enumerate(uv_from_json.astype(int)):
        cv2.circle(overlay, (u, v), point_radius, color, -1)
        if show_ids:
            label = f"{names[i]}"
            cv2.putText(overlay, label, (u + 6, v - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, thickness)

        if i > 0:
            u_prev, v_prev = uv_from_json[i-1].astype(int)
            cv2.line(overlay, (u_prev, v_prev), (u, v), color, thickness)

    title = f"{meta.get('view','?')} / {meta.get('cam','?')} | {os.path.basename(image_path)}"
    if reproj_errs is not None:
        title += f" | mean err: {reproj_errs.mean():.2f}px"

    plt.figure(figsize=(8, 6))
    plt.imshow(overlay)
    plt.axis("off")
    plt.title(title)
    plt.show()

# # 메인 실행 부분은 변경 없이 그대로 사용합니다.
# random_samples = random.sample(json_files, 5)
# for file_path in random_samples:
#     print(file_path)
#     visualize_json(file_path)

/home/najo/.conda/envs/dinov3/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/najo/.conda/envs/dinov3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 사용 가능한 GPU가 감지되었습니다.
GPU 개수: 3개
총 77466개의 JSON 파일을 찾았습니다.


In [2]:
def create_gt_heatmap(keypoint_2d, heatmap_size, sigma):
    H, W = heatmap_size
    x, y = keypoint_2d
    xx, yy = np.meshgrid(np.arange(W), np.arange(H))
    dist_sq = (xx - x)**2 + (yy - y)**2
    heatmap = np.exp(-dist_sq / (2 * sigma**2))
    heatmap[heatmap < np.finfo(float).eps * heatmap.max()] = 0
    return heatmap

def _scale_points(points_xy, from_size, to_size):
    Wf, Hf = from_size
    Wt, Ht = to_size
    out = np.empty_like(points_xy, dtype=np.float32)
    out[:, 0] = points_xy[:, 0] * (Wt / float(Wf))
    out[:, 1] = points_xy[:, 1] * (Ht / float(Hf))
    return out


In [3]:
class RobotPoseDataset(Dataset):
    def __init__(self, json_files, transform, sigma=5.0):
        self.json_files = json_files
        self.transform = transform
        self.sigma = sigma

    def __len__(self):
        return len(self.json_files)

    def __getitem__(self, idx):
        json_path = self.json_files[idx]

        with open(json_path, "r", encoding="utf-8") as f:
            sample = json.load(f)
        
        ###-------------- 이미지 관련 -------------###
        image_path = sample['meta']['image_path']
        img_bgr = cv2.imread(image_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]
        img_pil = Image.fromarray(img_rgb)
        image_dict = self.transform(img_pil)

        ###-------------- 히트맵 관련 -------------###
        Ht, Wt = (224, 224)
        keypoints = sample["objects"][0]["keypoints"]
        joint_num = len(keypoints)
        kpts_2d_orig = np.array([kp["projected_location"] for kp in keypoints])

        kpts_on_heatmap = _scale_points(kpts_2d_orig, from_size=(w, h), to_size=(Wt, Ht))
        heatmaps_np = np.zeros((joint_num, Ht, Wt), dtype=np.float32)
        for joint in range(joint_num):
            heatmaps_np[joint] = create_gt_heatmap(kpts_on_heatmap[joint], (Ht, Wt), self.sigma)
        gt_heatmaps_dict = torch.from_numpy(heatmaps_np)

        ###-------------- 관절각도 관련 -------------###
        angles = []
        for angle in sample['sim_state']["joints"]:
            angles.append(angle["position"])
        gt_angles = torch.tensor(angles, dtype=torch.float32)
        return image_dict, gt_heatmaps_dict, gt_angles

###----------- 관절개수 패딩 관련 -------------###
def robot_collate_fn_fixed(batch):
    images, heatmaps, angles = zip(*batch)
    images = torch.stack(images, 0)

    MAX_JOINTS = 7
    MAX_ANGLES = 9

    heatmaps_padded = torch.zeros(len(heatmaps), MAX_JOINTS, heatmaps[0].shape[1], heatmaps[0].shape[2])
    angles_padded = torch.zeros(len(angles), MAX_ANGLES)
    
    lengths = []
    for i, (h, a) in enumerate(zip(heatmaps, angles)):
        num_joints = h.shape[0]
        lengths.append(num_joints)
        heatmaps_padded[i, :num_joints, :, :] = h
        angles_padded[i, :a.shape[0]] = a

    lengths = torch.tensor(lengths, dtype=torch.long)
    return images, heatmaps_padded, angles_padded, lengths

In [4]:
FEATURE_DIM = 512
NUM_ANGLES = 9
NUM_JOINTS = 7

class DINOv3Backbone(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.model_name = model_name
        if "siglip" in model_name:
            self.model = SiglipVisionModel.from_pretrained(model_name)
        else:
            self.model = AutoModel.from_pretrained(model_name)
    def forward(self, image_tensor_batch):
        with torch.no_grad():
            if "siglip" in self.model_name:
                outputs = self.model(
                    pixel_values=image_tensor_batch,
                    interpolate_pos_encoding=True 
                )
                tokens = outputs.last_hidden_state
                patch_tokens = tokens[:, 1:, :]
            else: # DINOv3 계열
                outputs = self.model(pixel_values=image_tensor_batch)
                tokens = outputs.last_hidden_state
                num_reg = int(getattr(self.model.config, "num_register_tokens", 0))
                patch_tokens = tokens[:, 1 + num_reg :, :]
            return patch_tokens


# class JointAngleHead(nn.Module):
#     def __init__(self, input_dim=FEATURE_DIM, num_angles=NUM_ANGLES, num_queries=4, nhead=8, num_decoder_layers=2):
#         super().__init__()
        
#         self.pose_queries = nn.Parameter(torch.randn(1, num_queries, input_dim))
#         decoder_layer = nn.TransformerDecoderLayer(
#             d_model=input_dim, 
#             nhead=nhead, 
#             dim_feedforward=input_dim * 4, # 일반적인 설정
#             dropout=0.1, 
#             activation='gelu',
#             batch_first=True  # (batch, seq, feature) 입력을 위함
#         )
#         self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)
#         self.angle_predictor = nn.Sequential(
#             nn.LayerNorm(input_dim * num_queries),
#             nn.Linear(input_dim * num_queries, 512),
#             nn.GELU(),
#             nn.LayerNorm(512),
#             nn.Linear(512, 256),
#             nn.GELU(),
#             nn.LayerNorm(256),
#             nn.Linear(256, num_angles)
#         )
    
#     def forward(self, fused_features):
#         b = fused_features.size(0)
#         queries = self.pose_queries.repeat(b, 1, 1)
#         attn_output = self.transformer_decoder(tgt=queries, memory=fused_features)
#         output_flat = attn_output.flatten(start_dim=1)
#         return self.angle_predictor(output_flat)

class TokenFuser(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.projection = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.refine_blocks = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
    def forward(self, x):
        projected = self.projection(x)
        refined = self.refine_blocks(projected)
        residual = self.residual_conv(x)
        return torch.nn.functional.gelu(refined + residual)

class LightCNNStem(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1, bias=False), # 해상도 1/2
            nn.BatchNorm2d(16),
            nn.GELU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1, bias=False), # 해상도 1/4
            nn.BatchNorm2d(32),
            nn.GELU()
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1, bias=False), # 해상도 1/8
            nn.BatchNorm2d(64),
            nn.GELU()
        )
        
    def forward(self, x):
        feat_4 = self.conv_block1(x)  # 1/4 스케일 특징
        feat_8 = self.conv_block2(feat_4) # 1/8 스케일 특징
        return feat_4, feat_8 # 다른 해상도의 특징들을 반환

class FusedUpsampleBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.refine_conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU()
        )

    def forward(self, x, skip_feature):
        x = self.upsample(x)

        if x.shape[-2:] != skip_feature.shape[-2:]:
            skip_feature = F.interpolate(
                skip_feature, 
                size=x.shape[-2:], # target H, W
                mode='bilinear', 
                align_corners=False
            )

        fused = torch.cat([x, skip_feature], dim=1)
        return self.refine_conv(fused)

class UNetViTKeypointHead(nn.Module):
    def __init__(self, input_dim=768, num_joints=NUM_JOINTS, heatmap_size=(224, 224)):
        super().__init__()
        self.heatmap_size = heatmap_size
        self.token_fuser = TokenFuser(input_dim, 256)
        self.decoder_block1 = FusedUpsampleBlock(in_channels=256, skip_channels=64, out_channels=128)
        self.decoder_block2 = FusedUpsampleBlock(in_channels=128, skip_channels=32, out_channels=64)
        self.final_upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.heatmap_predictor = nn.Conv2d(64, num_joints, kernel_size=3, padding=1)

    def forward(self, dino_features, cnn_features):
        # if dist.get_rank() == 0: # 0번 GPU에서만 출력되도록 함
        #     print(f"DEBUG in Head: dino_features.shape = {dino_features.shape}")
        cnn_feat_4, cnn_feat_8 = cnn_features
        # num_patches_to_keep = 196
        # dino_features_sliced = dino_features[:, :num_patches_to_keep, :]
        
        # --- 수정된 부분 시작 ---
        b, n, d = dino_features.shape
        
        h = w = int(math.sqrt(n))
        
        if h * w != n:
            n_new = h * w
            dino_features = dino_features[:, :n_new, :]
        x = dino_features.permute(0, 2, 1).reshape(b, d, h, w)
        
        x = self.token_fuser(x)
        x = self.decoder_block1(x, cnn_feat_8)
        x = self.decoder_block2(x, cnn_feat_4)
        x = self.final_upsample(x)
        heatmaps = self.heatmap_predictor(x)
        
        return F.interpolate(heatmaps, size=self.heatmap_size, mode='bilinear', align_corners=False)
        
class DINOv3PoseEstimator(nn.Module):
    def __init__(self, dino_model_name, ablation_mode=None):
        super().__init__()
        self.ablation_mode = ablation_mode
        self.dino_model_name = dino_model_name
        self.backbone = DINOv3Backbone(dino_model_name)
        
        if "siglip" in self.dino_model_name:
            config = self.backbone.model.config
            feature_dim = config.hidden_size
        else: # DINOv3 계열
            config = self.backbone.model.config
            feature_dim = config.hidden_sizes[-1] if "conv" in self.dino_model_name else config.hidden_size
        
        self.cnn_stem = LightCNNStem()
        self.keypoint_head = UNetViTKeypointHead(input_dim=feature_dim)

    def forward(self, image_tensor_batch):
        dino_features = self.backbone(image_tensor_batch) # 항상 3D 텐서
        cnn_stem_features = self.cnn_stem(image_tensor_batch)

        if self.ablation_mode == 'cnn_only':
            dino_features = torch.zeros_like(dino_features)
        elif 'dino_only' in self.ablation_mode or 'siglip_only' in self.ablation_mode:
            cnn_stem_features = [torch.zeros_like(feat) for feat in cnn_stem_features]
        
        predicted_heatmaps = self.keypoint_head(dino_features, cnn_stem_features)
        return predicted_heatmaps

In [5]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image

###----------- 관절개수 패딩 관련 -------------###
def robot_collate_fn_fixed(batch):
    images, heatmaps, angles = zip(*batch)
    images = torch.stack(images, 0)

    MAX_JOINTS = 7
    MAX_ANGLES = 9

    heatmaps_padded = torch.zeros(len(heatmaps), MAX_JOINTS, heatmaps[0].shape[1], heatmaps[0].shape[2])
    angles_padded = torch.zeros(len(angles), MAX_ANGLES)
    
    lengths = []
    for i, (h, a) in enumerate(zip(heatmaps, angles)):
        num_joints = h.shape[0]
        lengths.append(num_joints)
        heatmaps_padded[i, :num_joints, :, :] = h
        angles_padded[i, :a.shape[0]] = a

    lengths = torch.tensor(lengths, dtype=torch.long)
    return images, heatmaps_padded, angles_padded, lengths

def get_max_preds(batch_heatmaps):
    '''
    히트맵 배치에서 최대값의 좌표를 찾는 함수
    batch_heatmaps: (batch_size, num_joints, height, width)
    '''
    assert isinstance(batch_heatmaps, torch.Tensor), 'batch_heatmaps should be torch.Tensor'
    assert batch_heatmaps.dim() == 4, 'batch_heatmaps should be 4-ndim'
    
    batch_size = batch_heatmaps.shape[0]
    num_joints = batch_heatmaps.shape[1]
    width = batch_heatmaps.shape[3]
    heatmaps_reshaped = batch_heatmaps.reshape(batch_size, num_joints, -1)
    maxvals, idx = torch.max(heatmaps_reshaped, 2)
    
    maxvals = maxvals.unsqueeze(-1)
    idx = idx.float()
    
    preds = torch.zeros((batch_size, num_joints, 2))
    preds[:, :, 0] = idx % width # x
    preds[:, :, 1] = torch.floor(idx / width) # y
    
    return preds, maxvals

# ======================= 시각화 헬퍼 함수 =======================
def denormalize(tensor, mean, std):
    """텐서를 역정규화하여 이미지로 복원합니다."""
    # clone()을 사용하여 원본 텐서가 변경되지 않도록 합니다.
    tensor = tensor.clone()
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return tensor

# ======================= 추론 및 시각화 실행 =======================
# --- 1. 기본 설정 ---

ablation_mode = 'dino_conv_only'
CHECKPOINT_PATH = f"checkpoints_{ablation_mode}/best_model.pth" # 불러올 모델 경로

# 검색을 시작할 최상위 폴더 경로
search_dir = "../dataset/Converted_dataset/TEST_SAMPLE"  # <-- 이곳에 실제 폴더 경로를 입력하세요.
json_files = glob.glob(os.path.join(search_dir, '**', '*.json'), recursive=True)

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 모드에 따라 모델 이름 결정
if 'vit' in ablation_mode:
    MODEL_NAME = 'facebook/dinov3-vitb16-pretrain-lvd1689m'
elif 'conv' in ablation_mode:
    MODEL_NAME = 'facebook/dinov3-convnext-base-pretrain-lvd1689m'
elif 'siglip' in ablation_mode:
    MODEL_NAME = 'google/siglip-base-patch16-224'
else: # cnn_only
    MODEL_NAME = 'facebook/dinov3-vitb16-pretrain-lvd1689m'

model = DINOv3PoseEstimator(dino_model_name=MODEL_NAME, ablation_mode=ablation_mode).to(device)
state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
model.load_state_dict(new_state_dict)
model.eval()
print(f"✅ 모델 로드 완료: {CHECKPOINT_PATH}")


for json_path in random.sample(json_files, min(13, len(json_files))):
    print(f"\n--- 처리 중: {os.path.basename(json_path)} ---")
    with open(json_path, "r", encoding="utf-8") as f:
        sample = json.load(f)

    # --- 이미지 준비 ---
    image_path = sample['meta']['image_path']
    try:
        img_bgr = cv2.imread(image_path)
        if img_bgr is None: raise FileNotFoundError
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        h, w = img_rgb.shape[:2]
        img_pil = Image.fromarray(img_rgb)
        image_tensor_input = transform(img_pil).unsqueeze(0).to(device)
    except Exception as e:
        print(f"🚨 이미지 처리 중 오류 발생 ({image_path}): {e}")
        continue

    # --- Ground Truth 데이터 준비 ---
    Ht, Wt = (224, 224)
    keypoints = sample["objects"][0]["keypoints"]
    joint_num_gt = len(keypoints)
    kpts_2d_orig = np.array([kp["projected_location"] for kp in keypoints])
    kpts_on_heatmap = _scale_points(kpts_2d_orig, from_size=(w, h), to_size=(Wt, Ht))
    heatmaps_np = np.zeros((joint_num_gt, Ht, Wt), dtype=np.float32)
    for i in range(joint_num_gt):
        heatmaps_np[i] = create_gt_heatmap(kpts_on_heatmap[i], (Ht, Wt), sigma=2.0)
    gt_heatmaps = torch.from_numpy(heatmaps_np)
    angles = [angle["position"] for angle in sample['sim_state']["joints"]]
    gt_angles_unpadded = torch.tensor(angles, dtype=torch.float32)
    
    dummy_batch = [(image_tensor_input[0].cpu(), gt_heatmaps, gt_angles_unpadded)]
    _, gt_heatmaps_padded, gt_angles_padded, _ = robot_collate_fn_fixed(dummy_batch)

    # --- 모델 추론 ---
    with torch.no_grad():
        pred_heatmaps = model(image_tensor_input)

    # --- 시각화 준비 ---
    predicted_heatmaps_for_viz = pred_heatmaps[0].cpu()
    combined_pred_heatmap = torch.sum(predicted_heatmaps_for_viz, dim=0).numpy()
    combined_gt_heatmap = torch.sum(gt_heatmaps_padded[0], dim=0).numpy()
    denorm_input_img = denormalize(image_tensor_input[0].cpu(), mean, std)
    denorm_input_img_np = denorm_input_img.numpy().transpose(1, 2, 0)
    denorm_input_img_np = np.clip(denorm_input_img_np, 0, 1)
    img_resized_for_heatmap = cv2.resize(denorm_input_img_np, (Wt, Ht))
    
    # <<< 추가된 부분: 예측 좌표와 GT 좌표 계산 >>>
    pred_coords, pred_maxvals = get_max_preds(pred_heatmaps)
    pred_coords_np = pred_coords[0].cpu().numpy()
    pred_maxvals_np = pred_maxvals[0].cpu().numpy() # 신뢰도 점수도 numpy로 변환
    
    # --- 플롯 그리기 ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # 서브플롯 1: 입력 이미지 + 예측(Red) & GT(Green) 키포인트
    axes[0].imshow(denorm_input_img_np)
    axes[0].set_title("Input with Predicted(🔴) & GT(🟢) Joints")
    axes[0].axis('off')
    
    scale_x_512 = 512 / w
    scale_y_512 = 512 / h
    scale_ht_224 = 224 / 512
    
    for i in range(joint_num_gt):
        # GT 키포인트 그리기 (녹색)
        x_gt, y_gt = kpts_2d_orig[i]
        axes[0].add_patch(plt.Circle((x_gt * scale_x_512, y_gt * scale_y_512), 6, color='lime', fill=False, linewidth=2))
        
        # 예측 키포인트 그리기 (빨간색)
        x_pred, y_pred = pred_coords_np[i]
        axes[0].add_patch(plt.Circle((x_pred / scale_ht_224, y_pred / scale_ht_224), 5, color='red'))
        score = pred_maxvals_np[i][0]
        label = f"{i}: {score:.2f}" # 예: "0: 0.98"
        axes[0].text(x_pred / scale_ht_224 + 8, y_pred / scale_ht_224 - 8, str(f'{i}'), color='white', fontsize=12, bbox=dict(facecolor='black', alpha=0.5))
        

    # 서브플롯 2: 예측 히트맵
    axes[1].imshow(img_resized_for_heatmap, alpha=0.6)
    axes[1].imshow(combined_pred_heatmap, cmap='jet', alpha=0.5)
    axes[1].set_title("Predicted Heatmaps")
    axes[1].axis('off')

    # 서브플롯 3: Ground Truth 히트맵
    axes[2].imshow(img_resized_for_heatmap, alpha=0.6)
    axes[2].imshow(combined_gt_heatmap, cmap='jet', alpha=0.5)
    axes[2].set_title("Ground Truth Heatmaps")
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()
    
    # --- 관절 각도 출력 ---
    # print("--- Joint Angle Comparison (Pred vs. GT) ---")
    # pred_a = pred_angles[0].cpu().numpy()
    # gt_a = gt_angles_unpadded.numpy() # 패딩되지 않은 원본 GT 각도 사용
    
    print(f"{'Joint':<5} | {'Predicted':<15} | {'Ground Truth':<15}")
    print("-" * 40)
    
    # <<< 수정된 부분: 실제 GT 각도 개수만큼만 반복 >>>
    # for i in range(len(gt_a)):
    #     print(f"J{i+1:<4} | {pred_a[i]:<15.4f} | {gt_a[i]:<15.4f}")

print("\n🎉 모든 샘플 이미지에 대한 추론 및 시각화 완료.")

/tmp/ipykernel_806301/877150075.py:94: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(CHECKPOINT_PATH, map_location=device)


✅ 모델 로드 완료: checkpoints_dino_conv_only/best_model.pth

--- 처리 중: zed_44377151_left_1747200975.986.json ---


RuntimeError: CUDA error: unknown error
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# # 1. 가중치를 담을 새로운 모델 구조를 정의합니다.
# #    UNetViTKeypointHead는 DINOv3의 특징 차원을 알아야 하므로 값을 직접 지정해줍니다.
# standalone_head = UNetViTKeypointHead(input_dim=768).to(device)

# # 2. 전체 state_dict 불러오기 (위와 동일)
# full_state_dict = torch.load("checkpoints/2nd_best_model_heatmap_only_epoch_98.pth")

# # 3. Keypoint Head에 해당하는 가중치만 필터링합니다.
# head_state_dict = {}
# for key, value in full_state_dict.items():
#     if key.startswith('keypoint_head.'):
#         # 'keypoint_head.' 접두사 제거
#         new_key = key.replace('keypoint_head.', '', 1)
#         head_state_dict[new_key] = value

# # 4. 필터링된 state_dict를 새로운 모델에 로드합니다.
# standalone_head.load_state_dict(head_state_dict)

# print("✅ Keypoint Head 모델만 성공적으로 로드했습니다.")